In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit

In [2]:
PROJECT_DIR = Path(
    r"C:\Users\User\Desktop\白血球画像分類"
)

METADATA_PATH = PROJECT_DIR / "metadata.csv"

metadata_df = pd.read_csv(METADATA_PATH)

metadata_df.head()

,subject,class_name,batch,montage_id,cell_id,row,col,ch1_path,ch3_path,ch5_path
0,CRF022,B,batch1,1,0,0,0,CRF022\B\batch1\montage_1\cell_0000_ch1.tif,CRF022\B\batch1\montage_1\cell_0000_ch3.tif,CRF022\B\batch1\montage_1\cell_0000_ch5.tif
1,CRF022,B,batch1,1,1,0,1,CRF022\B\batch1\montage_1\cell_0001_ch1.tif,CRF022\B\batch1\montage_1\cell_0001_ch3.tif,CRF022\B\batch1\montage_1\cell_0001_ch5.tif
2,CRF022,B,batch1,1,2,0,2,CRF022\B\batch1\montage_1\cell_0002_ch1.tif,CRF022\B\batch1\montage_1\cell_0002_ch3.tif,CRF022\B\batch1\montage_1\cell_0002_ch5.tif
3,CRF022,B,batch1,1,3,0,3,CRF022\B\batch1\montage_1\cell_0003_ch1.tif,CRF022\B\batch1\montage_1\cell_0003_ch3.tif,CRF022\B\batch1\montage_1\cell_0003_ch5.tif
4,CRF022,B,batch1,1,4,0,4,CRF022\B\batch1\montage_1\cell_0004_ch1.tif,CRF022\B\batch1\montage_1\cell_0004_ch3.tif,CRF022\B\batch1\montage_1\cell_0004_ch5.tif


In [3]:
subjects = metadata_df["subject"].unique()

print(subjects)
print("Number of subjects:", len(subjects))

<ArrowStringArray>
[     'CRF022',      'CRF034',      'CRF041',      'CRF049',      'CRF066',
      'CRF074', 'CRF101_2014', 'CRF101_2015',      'CRF102',      'CRF130',
      'CRF132',      'CRF186',       'CRF59']
Length: 13, dtype: str
Number of subjects: 13


<h3>Subject Groupを作る</h3>
<h4>同一人物と思われるデータをtrainとtestに分離しない</h4>

In [5]:
metadata_df["subject_group"] = (
    metadata_df["subject"]
    .str.replace(r"_\d{4}$", "", regex=True)
)
print(
    metadata_df[
        ["subject", "subject_group"]
    ]
    .drop_duplicates()
    .sort_values("subject_group")
)

           subject subject_group
0           CRF022        CRF022
4659        CRF034        CRF034
8302        CRF041        CRF041
14222       CRF049        CRF049
21721       CRF066        CRF066
29512       CRF074        CRF074
35113  CRF101_2014        CRF101
44327  CRF101_2015        CRF101
49796       CRF102        CRF102
62374       CRF130        CRF130
68044       CRF132        CRF132
80688       CRF186        CRF186
97171        CRF59         CRF59


<h3>Trainと一時データに分ける</h3>
<h4>7:3の比率</h4>

In [7]:
gss1 = GroupShuffleSplit(
    n_splits=1,
    #Make 70% of dataset as train dataset
    test_size=0.30,
    random_state=42
)
#Take one element in order
train_idx, temp_idx = next(
    gss1.split(
        metadata_df,
        groups=metadata_df["subject_group"]
    )
)

train_df = metadata_df.iloc[train_idx].copy()
temp_df = metadata_df.iloc[temp_idx].copy()

In [9]:
gss2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)
gss2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_idx, test_idx = next(
    gss2.split(
        temp_df,
        groups=temp_df["subject_group"]
    )
)

val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

val_idx, test_idx = next(
    gss2.split(
        temp_df,
        groups=temp_df["subject_group"]
    )
)

val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

In [10]:
#Add split label in each data
train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"
split_metadata_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

<h3>subjectが重複していないか確認</h3>

In [11]:
train_subjects = set(train_df["subject_group"].unique())
val_subjects = set(val_df["subject_group"].unique())
test_subjects = set(test_df["subject_group"].unique())

print("Train:", train_subjects)
print("Validation:", val_subjects)
print("Test:", test_subjects)

print(
    "Train ∩ Val:",
    train_subjects & val_subjects
)

print(
    "Train ∩ Test:",
    train_subjects & test_subjects
)

print(
    "Val ∩ Test:",
    val_subjects & test_subjects
)

Train: {'CRF101', 'CRF049', 'CRF066', 'CRF074', 'CRF59', 'CRF102', 'CRF034', 'CRF041'}
Validation: {'CRF022', 'CRF132'}
Test: {'CRF130', 'CRF186'}
Train ∩ Val: set()
Train ∩ Test: set()
Val ∩ Test: set()


<h3>各splitのクラス数を確認</h3>

In [13]:
split_summary = (
    split_metadata_df
    .groupby(["split", "class_name"])
    .size()
    .unstack(fill_value=0)
)

split_summary

class_name,B,T,eosinophil,monocyte,neutrophil
split,,,,,
test,800,4437,1215,1064,14637
train,2079,14104,2406,2231,39829
val,1249,6669,421,1051,7913


<h3>Datasetを保存</h3>

In [14]:
SPLIT_METADATA_PATH = (
    PROJECT_DIR / "metadata_with_split.csv"
)

split_metadata_df.to_csv(
    SPLIT_METADATA_PATH,
    index=False
)

print("Saved:")
print(SPLIT_METADATA_PATH)

Saved:
C:\Users\User\Desktop\白血球画像分類\metadata_with_split.csv
